# Gold Dim Date

Builds the `workspace.gold.dim_date` date dimension (Delta table) in pure SQL, covering 2010-01-01 through 2014-12-31. The `date_key` is an INT surrogate key in YYYYMMDD format, matching `fact_sales.order_date_key`, `ship_date_key`, and `due_date_key`.

In [0]:
%sql
-- Gold Dim Date
-- Date dimension for the gold layer: 2010-01-01 to 2014-12-31 (covers all fact_sales dates 2010-12-29..2014-02-09).
-- date_key is an INT in YYYYMMDD format, matching fact_sales.order_date_key / ship_date_key / due_date_key.
-- CREATE OR REPLACE handles the schema left behind by the previous (deleted) dim_date notebook.

CREATE OR REPLACE TABLE workspace.gold.dim_date AS
WITH dates AS (
  SELECT explode(sequence(date '2010-01-01', date '2014-12-31', interval 1 day)) AS full_date
)
SELECT
  CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
  full_date,
  year(full_date)          AS year,
  quarter(full_date)       AS quarter,
  month(full_date)         AS month,
  monthname(full_date)     AS month_name,
  dayofweek(full_date)     AS day_of_week,   -- 1 = Sunday, 7 = Saturday
  day(full_date)           AS day_of_month,
  dayofyear(full_date)     AS day_of_year,
  dayname(full_date)       AS day_name,
  weekofyear(full_date)    AS week_of_year,
  date_format(full_date, 'yyyy-MM') AS year_month,
  CASE WHEN dayofweek(full_date) IN (1, 7) THEN false ELSE true END AS is_weekday,
  dayofweek(full_date) IN (1, 7) AS is_weekend
FROM dates;

In [0]:
%sql
-- Verify dim_date: row count, key range, and sample rows

SELECT COUNT(*) AS row_count, MIN(date_key) AS min_date_key, MAX(date_key) AS max_date_key
FROM workspace.gold.dim_date;

SELECT * FROM workspace.gold.dim_date ORDER BY full_date LIMIT 5;